# NN-03 Training Loop

- Module: 06 Neural Networks
- Topic: training a scratch NumPy MLP on a synthetic nonlinear classification problem
- Estimated runtime: 6 minutes
- Prerequisites: forward pass, backpropagation, and gradient descent updates
- Expected outputs: training curves, train/test accuracy, and a decision-boundary plot on a noisy XOR dataset


## Learning Goals

By the end of this notebook, you should be able to:
- train a small neural network with repeated forward and backward passes;
- monitor loss and accuracy over epochs; and
- modify `layer_dims` or `activations` to see how architecture choices affect performance.


In [ ]:
from __future__ import annotations

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import numpy as np
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('Could not locate repo root from notebook path.')


REPO_ROOT = find_repo_root(Path.cwd())
MODULE_SRC = REPO_ROOT / 'modules/06-neural-networks/src'
if str(MODULE_SRC) not in sys.path:
    sys.path.insert(0, str(MODULE_SRC))

from nn_from_scratch import ScratchMLP, train_test_split_indices

SEED = 7
rng = np.random.default_rng(SEED)
SEED



## Outline

1. Generate a synthetic XOR-style dataset that is not linearly separable.
2. Choose a small network architecture.
3. Train with gradient descent.
4. Visualize learning curves and the learned decision boundary.


## Step 1 - Create a synthetic nonlinear classification problem

The labels depend on the sign pattern of the two coordinates, so a single linear separator cannot solve the task. This keeps the lab self-contained while still requiring a hidden layer.


In [ ]:
def make_noisy_xor(sample_size: int = 200, noise_std: float = 0.15, seed: int = SEED):
    local_rng = np.random.default_rng(seed)
    X = local_rng.uniform(-1.0, 1.0, size=(sample_size, 2))
    y = ((X[:, 0] * X[:, 1]) < 0.0).astype(int)
    X = X + noise_std * local_rng.normal(size=X.shape)
    return X, y


X_raw, y = make_noisy_xor()
X = (X_raw - X_raw.mean(axis=0, keepdims=True)) / X_raw.std(axis=0, keepdims=True)
train_idx, test_idx = train_test_split_indices(len(X), test_fraction=0.25, seed=SEED)
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

fig, ax = plt.subplots(figsize=(5.4, 4.4))
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='coolwarm', s=22, edgecolor='black', linewidth=0.2)
ax.set_title('Noisy XOR training set')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
fig.tight_layout()
plt.close(fig)
fig


## Step 2 - Expose the architecture knobs

Change `layer_dims` to add or remove hidden layers. Change `activations` to swap `tanh` for `relu`. The only rule is one activation name per affine layer.


In [ ]:
layer_dims = [2, 12, 1]
activations = ['tanh', 'sigmoid']
learning_rate = 0.20
epochs = 800
record_every = 100

{
    'layer_dims': layer_dims,
    'activations': activations,
    'learning_rate': learning_rate,
    'epochs': epochs,
}


## Step 3 - Train the model

Each epoch does four things: forward pass, loss computation, backward pass, and parameter update. The history object stores checkpoints so we can inspect convergence without printing every epoch.


In [ ]:
model = ScratchMLP(layer_dims=layer_dims, activations=activations, loss='binary_cross_entropy', seed=SEED)
history = model.fit(
    X_train,
    y_train,
    epochs=epochs,
    learning_rate=learning_rate,
    record_every=record_every,
)

metrics = {
    'final_train_loss': round(history['loss'][-1], 4),
    'final_train_accuracy': round(history['accuracy'][-1], 4),
    'test_accuracy': round(model.accuracy(X_test, y_test), 4),
}
metrics


## Step 4 - Inspect the learning curves

A healthy run should show loss trending downward while training accuracy rises toward a stable plateau.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.4))
axes[0].plot(history['epoch'], history['loss'], marker='o', color='tab:blue')
axes[0].set_title('Loss during training')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('binary cross-entropy')

axes[1].plot(history['epoch'], history['accuracy'], marker='o', color='tab:green')
axes[1].set_title('Training accuracy')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('accuracy')
axes[1].set_ylim(0.0, 1.05)

fig.tight_layout()
plt.close(fig)
fig


## Step 5 - Plot the learned decision boundary

The hidden layer bends the classifier away from a single straight line. This is the visible payoff of composing affine maps with nonlinearities.


In [ ]:
grid_x1, grid_x2 = np.meshgrid(
    np.linspace(X[:, 0].min() - 0.4, X[:, 0].max() + 0.4, 220),
    np.linspace(X[:, 1].min() - 0.4, X[:, 1].max() + 0.4, 220),
)
grid = np.column_stack([grid_x1.ravel(), grid_x2.ravel()])
grid_probs = model.predict_proba(grid).reshape(grid_x1.shape)

fig, ax = plt.subplots(figsize=(5.4, 4.4))
contour = ax.contourf(grid_x1, grid_x2, grid_probs, levels=np.linspace(0.0, 1.0, 11), cmap='coolwarm', alpha=0.75)
ax.contour(grid_x1, grid_x2, grid_probs, levels=[0.5], colors='black', linewidths=1.5)
ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='coolwarm', s=28, edgecolor='black', linewidth=0.3)
ax.set_title('Decision boundary on held-out data')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
fig.colorbar(contour, ax=ax, label='predicted probability of class 1')
fig.tight_layout()
plt.close(fig)
fig


## Extension Prompt

Try one change at a time:
- reduce the hidden width to `4` and compare the final accuracy;
- add a second hidden layer with `layer_dims = [2, 8, 4, 1]`;
- replace `tanh` with `relu` and see whether convergence becomes faster or less stable.


In [ ]:
# Example scaffold for a deeper architecture. Uncomment to compare runs.
# deeper_model = ScratchMLP(layer_dims=[2, 8, 4, 1], activations=['tanh', 'tanh', 'sigmoid'], loss='binary_cross_entropy', seed=SEED)
# deeper_history = deeper_model.fit(X_train, y_train, epochs=epochs, learning_rate=learning_rate, record_every=record_every)
# {'deeper_test_accuracy': deeper_model.accuracy(X_test, y_test)}


## Summary

The same reusable code path handled forward propagation, backpropagation, and parameter updates. On a nonlinear synthetic dataset, the scratch network reached strong train and test accuracy without relying on a deep-learning framework.
